# Practical: Food Delivery Help Desk

This notebook implements the practical assignment:
- **Task 1: Manager (5 marks)**: Two worker agents (`delivery`, `payment`) exposed as tools via `.as_tool()`, managed by a central manager agent.
- **Task 2: Blackboard (5 marks)**: Workflow (`start` -> `allergy_step` -> `summary_step`) using a shared blackboard state.


## Step 0: Setup
Connect to Groq and define helpers.

In [1]:
import os
from dotenv import load_dotenv, find_dotenv
from agent_framework import WorkflowBuilder, WorkflowContext, executor
from agent_framework.openai import OpenAIChatCompletionClient

load_dotenv(find_dotenv(usecwd=True))

# One connection to Groq (OpenAI-compatible)
groq = OpenAIChatCompletionClient(
    model="openai/gpt-oss-20b",
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)

SETTINGS = {"temperature": 0}


def make_agent(name, instructions, tools=None):
    if tools:
        return groq.as_agent(name=name, instructions=instructions,
                             tools=tools, default_options=SETTINGS)
    return groq.as_agent(name=name, instructions=instructions,
                         default_options=SETTINGS)


async def ask(agent, text):
    reply = await agent.run(text)
    return reply.text.strip()


def who_was_called(reply):
    names = []
    for message in reply.messages:
        for item in message.contents:
            if item.type == "function_call":
                names.append(item.name)
    return names


print("Setup done")


Setup done


---
# Task 1: Manager (5 marks)

1. Make two worker agents, `delivery` and `payment`.
2. Turn each one into a tool with `as_tool`.
3. Build a `manager` agent that uses both tools.
4. Run the manager on: `"My food showed up an hour after the promised time"`

**Expected result:** `Task 1: [ask_delivery]`

In [2]:
# 1. Worker agents
delivery_agent = make_agent(
    "delivery",
    "You are a delivery specialist. You handle late food, delivery delays, and driver issues. Reply in one short sentence."
)

payment_agent = make_agent(
    "payment",
    "You are a payment specialist. You handle charges, payment methods, billing, and refunds. Reply in one short sentence."
)

# 2. Turn each worker into a tool with as_tool
delivery_tool = delivery_agent.as_tool(
    name="ask_delivery",
    description="Ask the delivery specialist about delivery status, late food, and driver delays."
)

payment_tool = payment_agent.as_tool(
    name="ask_payment",
    description="Ask the payment specialist about charges, payment issues, billing, and refunds."
)

# 3. Manager agent using both tools
manager = make_agent(
    "manager",
    "You are a support manager. Never answer yourself. "
    "Pick the right specialist, call their tool, and return their answer.",
    tools=[delivery_tool, payment_tool],
)

print("Manager ready with delivery and payment tools")


Manager ready with delivery and payment tools


In [3]:
# Run manager on the ticket
ticket = "My food showed up an hour after the promised time"
reply = await manager.run(ticket)

print("Task 1:", who_was_called(reply))
print("Answer:", reply.text)


Task 1: ['ask_delivery']
Answer: Apologies for the delay; I’ll investigate and get your food to you ASAP.


---
# Task 2: Blackboard (5 marks)

Build workflow: `start` -> `allergy_step` -> `summary_step`
- `start` saves the ticket and an empty `"notes"` list on the board.
- `allergy_step` reads the notes, adds `"allergy: ALLERGY"` or `"allergy: OK"`, and writes them back.
- `summary_step` prints the ticket and the notes.

Run on: `"I have a nut allergy and my salad had peanuts in it!"`

**Expected result:** `Notes: allergy: ALLERGY`

In [4]:
# Allergy specialist agent
allergy_agent = make_agent(
    "allergy",
    "You are an allergy checker. Reply with one word: ALLERGY if the text mentions food allergies, nuts, ingredients, or allergic reactions, otherwise OK."
)


# Executors: start -> allergy_step -> summary_step
@executor(id="start")
async def start(ticket: str, ctx: WorkflowContext[str]) -> None:
    ctx.set_state("ticket", ticket)     # save the ticket on the board
    ctx.set_state("notes", [])          # save an empty "notes" list on the board
    await ctx.send_message("go")


@executor(id="allergy_step")
async def allergy_step(_: str, ctx: WorkflowContext[str]) -> None:
    ticket = ctx.get_state("ticket")
    notes = ctx.get_state("notes")                  # 1. READ the notes
    answer = await ask(allergy_agent, ticket)
    notes = notes + ["allergy: " + answer]          # 2. ADD "allergy: ALLERGY" or "allergy: OK"
    ctx.set_state("notes", notes)                   # 3. WRITE them back
    await ctx.send_message("go")


@executor(id="summary_step")
async def summary_step(_: str, ctx: WorkflowContext) -> None:
    ticket = ctx.get_state("ticket")
    notes = ctx.get_state("notes")
    print("Ticket:", ticket)
    print("Notes:", ", ".join(notes))
    await ctx.yield_output("Ticket: " + ticket + "\nNotes: " + ", ".join(notes))


# Build workflow
board_flow = (WorkflowBuilder(start_executor=start)
              .add_edge(start, allergy_step)
              .add_edge(allergy_step, summary_step)
              .build())

print("Workflow built successfully")


Workflow built successfully


In [5]:
# Run Blackboard workflow
allergy_ticket = "I have a nut allergy and my salad had peanuts in it!"
result = await board_flow.run(allergy_ticket)


Ticket: I have a nut allergy and my salad had peanuts in it!
Notes: allergy: ALLERGY
